# 부품 누락 판별기 단독 학습·검증

**L4/A100 새 런타임 → 기존 Secret GITHUB_TOKEN 허용 → 모두 실행.** Drive 인증 외 수동 라벨·박스·경로 입력은 없습니다. 기존 Drive 원본 ZIP과 dev QA를 자동으로 찾습니다.

기존 glass-insulator train 정상/부품 누락 라벨로 **고정 DINOv2 특징 + 작은 지도학습 분류기**를 학습합니다. DINO 본체는 학습하지 않으며 전체 사진을 패딩·518 리사이즈해 중앙 crop을 피합니다. 기존 LVLM 결과·Q1/Q2/Q3 답은 그대로 보존합니다.

1. 공식 val/test + dev240 + gold246 사진을 이름·크기/CRC·바이트 SHA·RGB 픽셀 SHA로 제외합니다.
2. 남은 train을 **촬영일별 fit / calibration / internal_test**로 분리합니다.
3. fit으로만 분류기를 학습하고 calibration으로만 임계값을 정합니다(정상 오탐5% 이하에서 균형 정확도 최대).
4. 모델·임계값 저장 후 내부 평가 및 기존 dev50장 성능을 보고합니다. **gold는 평가하지 않습니다.**

새 실행마다 시간+UUID 전용 폴더를 사용합니다. 같은 실행 셀을 다시 돌리면 성공 특징을 재사용하며, 입력/코드가 바뀌면 중단합니다. 새 런타임의 모두 실행은 새 결과로 보존합니다. 실패한 실행은 완료 ZIP으로 내보내지 않습니다.

이 실험은 지도학습 보조라는 별도 조건이며 실제 효과는 실행 후 확인합니다. 기존 dev는 탐색용이며 다른 촬영일·새 장비·다른 결함 성능을 보장하지 않습니다. 정확한 픽셀 중복은 차단하지만 다른 crop·손실 재압축 등 모든 유사 사진을 증명한 것은 아닙니다.

출력: `MyDrive/lvlm_insplad_results/missing_component_probe_v1/<시간_UUID>/`.
마지막에 **missing_component_probe_results.zip**이 다운로드됩니다. GPU 특징 추출은 아직 실제 실행 전입니다.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
REPO = "arnold6444/lvlm_insplad"
EXECUTION_COMMIT = "cc75275dd69f3cf612c0d3295a06a0b9c8408c9f"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
RUN_FOLDER = "lvlm_insplad_results/missing_component_probe_v1"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
SEED = 42
print("이번 결과 폴더:", RUN_FOLDER + "/" + RUN_ID)


In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER / RUN_ID
if not RUN_ROOT.parent.name.startswith("missing_component_probe_v1"):
    raise ValueError("새 전용 결과 폴더가 필요합니다.")
REPO_DIR = Path("/content/missing_component_probe_code")

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
# Fetch exactly the locked object, never a moving branch.
if not REPO_DIR.exists():
    REPO_DIR.mkdir(parents=True)
    run(["git", "init", REPO_DIR])
    run(["git", "-C", REPO_DIR, "remote", "add", "origin", f"https://github.com/{REPO}.git"])
    run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", EXECUTION_COMMIT], env=git_env)
    run(["git", "-C", REPO_DIR, "checkout", "--detach", EXECUTION_COMMIT])
else:
    existing_remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    if existing_remote != f"https://github.com/{REPO}.git":
        raise RuntimeError("실행 폴더의 저장소가 다릅니다. 새 런타임을 사용하세요.")
actual_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if actual_revision != EXECUTION_COMMIT:
    raise RuntimeError("고정 커밋과 다른 코드입니다. 새 런타임을 사용하세요.")
if subprocess.check_output(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, text=True).strip():
    raise RuntimeError("고정 코드가 수정됐습니다. 새 런타임을 사용하세요.")
del auth, git_env
run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.14.0",
     "scikit-learn==1.7.2", "numpy==2.2.6", "scipy==1.16.3", "pillow==11.3.0", "pandas==2.3.3"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
from verify_gold_revision import verify as verify_gold_revision
verify_gold_revision(REPO_DIR, EXECUTION_COMMIT)
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")
if torch.cuda.get_device_properties(0).total_memory < 20 * 1024**3:
    raise RuntimeError("VRAM 20GB 이상 L4/A100이 필요합니다.")


In [ ]:
def find_zip(filename, expected_size):
    candidates = [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(filename, "Drive 자동 검색 중", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"Drive에서 {filename} ({expected_size} bytes)를 찾을 수 없습니다.")

# Remote ZIP random reads are slow; use local copies and verify content equality.
import hashlib
DATA_DIR = Path("/content/missing_component_probe_data")
DATA_DIR.mkdir(exist_ok=True)
def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(1024*1024), b""):
            h.update(b)
    return h.hexdigest()
def local_copy(source):
    dest = DATA_DIR / source.name
    source_sha = sha256(source)
    if not dest.exists() or sha256(dest) != source_sha:
        print("원본 ZIP 로컬 복사:", source.name, flush=True)
        tmp = dest.with_suffix(".copying")
        shutil.copyfile(source, tmp)
        if sha256(tmp) != source_sha:
            raise RuntimeError("ZIP 복사 SHA256 불일치")
        tmp.replace(dest)
    return dest
SUP_ZIP = local_copy(find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE))
UNSUP_ZIP = local_copy(find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE))
summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN":
    raise RuntimeError("FROZEN dev 보호 목록이 필요합니다.")
DEV_QA = DEV_DIR / summary["file"]
# Protection-only gold list is fixed in the execution commit. No gold inference.
GOLD_DIR = REPO_DIR / "results/latest/gold_v2_final"
GOLD_CSV = GOLD_DIR / "manual_review_final.csv"
if json.loads((GOLD_DIR / "build_summary.json").read_text(encoding="utf-8")).get("quality_status") != "FROZEN":
    raise RuntimeError("FROZEN gold 보호 목록이 필요합니다.")
for path in [DEV_QA, GOLD_CSV]:
    if not path.is_file():
        raise FileNotFoundError(path)
# Dev identity must match the committed, known cohort (not an arbitrary changed QA).
LOCKED_DEV_QA = REPO_DIR / "results/latest/gold_v2_dev/dev_master_qa_480.jsonl"
if sha256(DEV_QA) != sha256(LOCKED_DEV_QA):
    raise RuntimeError("기존 dev240 QA와 다릅니다. 임의로 학습을 계속하지 않습니다.")
print("dev는 확정 후 보고만 / gold는 제외 목록으로만 사용합니다.")
print("새 결과 폴더:", RUN_ROOT)


In [ ]:
# 실패하면 이 셀을 다시 실행하세요. 같은 입력의 성공 특징을 재사용합니다.
verify_gold_revision(REPO_DIR, EXECUTION_COMMIT)
run([sys.executable, REPO_DIR / "scripts/run_missing_component_probe.py",
     "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP, "--dev-qa", DEV_QA,
     "--gold-csv", GOLD_CSV, "--out-dir", RUN_ROOT, "--seed", str(SEED)])


In [ ]:
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
meta = json.loads((RUN_ROOT / "run_metadata.json").read_text(encoding="utf-8"))
if not status.get("complete") or not meta.get("complete") or status.get("row_errors", 1):
    raise RuntimeError("미완료 실행입니다. 위 오류를 해결하고 실행 셀을 다시 돌리세요.")
if meta["benchmark_commit"] != EXECUTION_COMMIT or meta["gold_scored"]:
    raise RuntimeError("실행 버전/평가 범위 불일치")
if sha256(RUN_ROOT / "frozen_detector.json") != status["frozen_detector_sha256"]:
    raise RuntimeError("확정 모델이 변경됐습니다.")
audit = json.loads((RUN_ROOT / "leakage_audit.json").read_text(encoding="utf-8"))
assert audit["exact_byte_overlap"] == audit["exact_rgb_overlap"] == audit["internal_date_overlap"] == 0
print("학습 누출 검사:", audit)
print("처리 시간(초):", meta["wall_time_s"], "GPU:", meta["gpu"])
metrics = pd.read_csv(RUN_ROOT / "leaderboard.csv")
display(metrics[["split", "images", "normal", "missing_cap", "tp", "fp", "tn", "fn",
                 "recall", "normal_fpr", "f1", "balanced_accuracy", "roc_auc", "average_precision"]])
print("internal_test가 촬영일 분리 평가입니다. dev_report_only는 기존 dev50장 탐색 보고입니다.")
print("오탐과 누락을 함께 확인하세요. Q1/Q2/Q3 답은 바꾸지 않았고 gold는 실행하지 않았습니다.")
print("결과:", RUN_ROOT)


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
# A frozen fit alone is not a successful evaluation. Export only complete runs.
if not json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8")).get("complete"):
    raise RuntimeError("실패/미완료 결과는 완료 ZIP으로 내보내지 않습니다.")
archive = Path("/content/missing_component_probe_results.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for path in sorted(RUN_ROOT.iterdir()):
        if path.is_file() and path.suffix != ".tmp":
            z.write(path, path.name)
files.download(str(archive))
